# Model Training 03c (Benchmark-Derived)

This notebook is intentionally simple and benchmark-aligned:

1. use only 4 base features (`swir22`, `NDMI`, `MNDWI`, `pet`),
2. train 3 separate Random Forest regressors,
3. evaluate with a random 70/30 split,
4. generate a submission from merged MVP validation parquet.

In [1]:
import os
from datetime import datetime

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, HistGradientBoostingRegressor
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.compose import TransformedTargetRegressor
from sklearn.metrics import r2_score, mean_squared_error
from joblib import Parallel, delayed
from tqdm.auto import tqdm

In [2]:
TRAIN_PATH = '../data/interim/water_quality_mvp_baseline.parquet'
VALID_PATH = '../data/interim/water_quality_mvp_validation.parquet'
TEMPLATE_PATH = '../data/raw/submission_template.csv'

FEATURES = ['swir22', 'NDMI', 'MNDWI', 'pet']
TARGETS = ['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']
META = ['Longitude', 'Latitude', 'Sample Date']

train_df = pd.read_parquet(TRAIN_PATH).copy()
val_df = pd.read_parquet(VALID_PATH).copy()
template_df = pd.read_csv(TEMPLATE_PATH).copy()

required_train = FEATURES + TARGETS
missing_train = [c for c in required_train if c not in train_df.columns]
if missing_train:
    raise RuntimeError(f'Missing training columns: {missing_train}')

missing_val = [c for c in FEATURES if c not in val_df.columns]
if missing_val:
    raise RuntimeError(f'Missing validation columns: {missing_val}')

print('Train shape:', train_df.shape)
print('Validation shape:', val_df.shape)
print('Template shape:', template_df.shape)

Train shape: (9319, 10)
Validation shape: (200, 10)
Template shape: (200, 6)


In [3]:
# Median imputation aligned with benchmark notebook behavior
train_medians = train_df[FEATURES].median(numeric_only=True)
val_medians = val_df[FEATURES].median(numeric_only=True)

X_full = train_df[FEATURES].copy().fillna(train_medians)
Y_full = train_df[TARGETS].copy()
X_sub = val_df[FEATURES].copy().fillna(val_medians)

print('Feature null counts (train):')
print(X_full.isna().sum())
print('\nFeature null counts (validation):')
print(X_sub.isna().sum())

Feature null counts (train):
swir22    0
NDMI      0
MNDWI     0
pet       0
dtype: int64

Feature null counts (validation):
swir22    0
NDMI      0
MNDWI     0
pet       0
dtype: int64


In [4]:
def split_data(X, y, test_size=0.3, random_state=42):
    return train_test_split(X, y, test_size=test_size, random_state=random_state)


def scale_data(X_train, X_test):
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    return X_train_scaled, X_test_scaled, scaler


def evaluate_model(model, X_scaled, y_true, dataset_name='Test', verbose=True):
    y_pred = model.predict(X_scaled)
    r2 = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    if verbose:
        print(f'\n{dataset_name} Evaluation:')
        print(f'R2: {r2:.3f}')
        print(f'RMSE: {rmse:.3f}')
    return y_pred, r2, rmse


def instantiate_model(candidate):
    kind = candidate['kind']
    params = candidate['params'].copy()
    if kind == 'rf':
        base = RandomForestRegressor(**params)
    elif kind == 'et':
        base = ExtraTreesRegressor(**params)
    elif kind == 'hgb':
        base = HistGradientBoostingRegressor(**params)
    elif kind == 'ridge':
        base = Ridge(**params)
    elif kind == 'elastic':
        base = ElasticNet(**params)
    else:
        raise ValueError(f'Unknown model kind: {kind}')

    if candidate.get('log_target', False):
        return TransformedTargetRegressor(
            regressor=base,
            func=np.log1p,
            inverse_func=np.expm1,
        )

    return base


def run_single_split(X, y, candidate, random_state=42, verbose=False):
    X_train, X_test, y_train, y_test = split_data(X, y, random_state=random_state)
    X_train_scaled, X_test_scaled, scaler = scale_data(X_train, X_test)

    model = instantiate_model(candidate)
    model.fit(X_train_scaled, y_train)

    _, r2_train, rmse_train = evaluate_model(model, X_train_scaled, y_train, 'Train', verbose=verbose)
    _, r2_test, rmse_test = evaluate_model(model, X_test_scaled, y_test, 'Test', verbose=verbose)

    return {
        'R2_Train': r2_train,
        'RMSE_Train': rmse_train,
        'R2_Test': r2_test,
        'RMSE_Test': rmse_test,
    }


def score_candidate(X, y, candidate, seeds):
    rows = []
    for seed in seeds:
        out = run_single_split(X, y, candidate, random_state=seed, verbose=False)
        rows.append({
            'seed': seed,
            'R2_Test': out['R2_Test'],
            'RMSE_Test': out['RMSE_Test'],
        })

    split_df = pd.DataFrame(rows)
    mean_r2 = float(split_df['R2_Test'].mean())
    std_r2 = float(split_df['R2_Test'].std(ddof=0))
    mean_rmse = float(split_df['RMSE_Test'].mean())
    robust_score = float(mean_r2 - 0.25 * std_r2)
    return split_df, mean_r2, std_r2, mean_rmse, robust_score


def evaluate_candidate_row(X, y, target, candidate, seeds):
    _, mean_r2, std_r2, mean_rmse, robust_score = score_candidate(X, y, candidate, seeds)
    return {
        'Parameter': target,
        'candidate_id': candidate['candidate_id'],
        'kind': candidate['kind'],
        'Mean_R2_Test': mean_r2,
        'Std_R2_Test': std_r2,
        'Mean_RMSE_Test': mean_rmse,
        'Robust_Score': robust_score,
    }


def fit_full_model(X, y, candidate):
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    model = instantiate_model(candidate)
    model.fit(X_scaled, y)
    return model, scaler

In [5]:
BASELINE_SPEC = {
    'candidate_id': 'rf_baseline',
    'kind': 'rf',
    'params': {
        'n_estimators': 100,
        'random_state': 42,
        'n_jobs': -1,
    },
}
SEED_SWEEP = [42, 52, 62, 72]
DRP_ALPHAS = [0.50, 0.35]

# Hyperparameter sweep is intentionally disabled in this notebook revision.
# We keep rf_baseline as the fixed training anchor and focus on DRP hedge variants.

models = {}
scalers = {}
baseline_models = {}
baseline_scalers = {}
results = []

for target in TARGETS:
    y = Y_full[target]

    split_rows = []
    for seed in SEED_SWEEP:
        out = run_single_split(X_full, y, BASELINE_SPEC, random_state=seed, verbose=False)
        split_rows.append({
            'seed': seed,
            'R2_Test': out['R2_Test'],
            'RMSE_Test': out['RMSE_Test'],
        })

    split_df = pd.DataFrame(split_rows)
    mean_r2 = float(split_df['R2_Test'].mean())
    std_r2 = float(split_df['R2_Test'].std(ddof=0))
    mean_rmse = float(split_df['RMSE_Test'].mean())
    robust_score = float(mean_r2 - 0.25 * std_r2)

    model, scaler = fit_full_model(X_full, y, BASELINE_SPEC)
    models[target] = model
    scalers[target] = scaler
    baseline_models[target] = model
    baseline_scalers[target] = scaler

    print(f"\nTarget: {target}")
    print(f"Baseline mean R2={mean_r2:.6f}, std R2={std_r2:.6f}, mean RMSE={mean_rmse:.6f}")

    results.append({
        'Parameter': target,
        'Baseline_Candidate': BASELINE_SPEC['candidate_id'],
        'Baseline_Mean_R2_Test': mean_r2,
        'Best_Candidate': BASELINE_SPEC['candidate_id'],
        'Best_Mean_R2_Test': mean_r2,
        'Selected_Candidate': BASELINE_SPEC['candidate_id'],
        'Selected_Mean_R2_Test': mean_r2,
        'Selected_Std_R2_Test': std_r2,
        'Selected_Mean_RMSE_Test': mean_rmse,
        'Selected_Robust_Score': robust_score,
        'Selection_Reason': 'baseline_only_no_tuning',
    })

results_summary = pd.DataFrame(results)
results_summary



Target: Total Alkalinity
Baseline mean R2=0.536568, std R2=0.008823, mean RMSE=50.996372

Target: Electrical Conductance
Baseline mean R2=0.588000, std R2=0.002926, mean RMSE=219.359493

Target: Dissolved Reactive Phosphorus
Baseline mean R2=0.538335, std R2=0.009622, mean RMSE=34.991196


,Parameter,Baseline_Candidate,Baseline_Mean_R2_Test,Best_Candidate,Best_Mean_R2_Test,Selected_Candidate,Selected_Mean_R2_Test,Selected_Std_R2_Test,Selected_Mean_RMSE_Test,Selected_Robust_Score,Selection_Reason
0,Total Alkalinity,rf_baseline,0.536568,rf_baseline,0.536568,rf_baseline,0.536568,0.008823,50.996372,0.534362,baseline_only_no_tuning
1,Electrical Conductance,rf_baseline,0.588000,rf_baseline,0.588000,rf_baseline,0.588000,0.002926,219.359493,0.587269,baseline_only_no_tuning
2,Dissolved Reactive Phosphorus,rf_baseline,0.538335,rf_baseline,0.538335,rf_baseline,0.538335,0.009622,34.991196,0.535930,baseline_only_no_tuning


In [ ]:
if len(template_df) != len(val_df):
    raise RuntimeError(f'Row count mismatch: template={len(template_df)} validation={len(val_df)}')

template_dates = pd.to_datetime(template_df['Sample Date'], dayfirst=True, errors='coerce')
val_dates = pd.to_datetime(val_df['Sample Date'], errors='coerce')
lon_ok = np.allclose(template_df['Longitude'].to_numpy(float), val_df['Longitude'].to_numpy(float), atol=1e-9)
lat_ok = np.allclose(template_df['Latitude'].to_numpy(float), val_df['Latitude'].to_numpy(float), atol=1e-9)
date_ok = template_dates.equals(val_dates)

if not (lon_ok and lat_ok and date_ok):
    raise RuntimeError('Template and validation row order mismatch. Abort submission build.')


def predict_bundle(model_dict, scaler_dict, X_data):
    out = {}
    for target in TARGETS:
        X_scaled = scaler_dict[target].transform(X_data)
        pred = model_dict[target].predict(X_scaled)
        out[target] = np.clip(np.asarray(pred, dtype=float), 0, None)
    return out


def make_submission(ta_vals, ec_vals, drp_vals):
    return pd.DataFrame({
        'Longitude': template_df['Longitude'].values,
        'Latitude': template_df['Latitude'].values,
        'Sample Date': template_df['Sample Date'].values,
        'Total Alkalinity': ta_vals,
        'Electrical Conductance': ec_vals,
        'Dissolved Reactive Phosphorus': drp_vals,
    })


preds_anchor = predict_bundle(baseline_models, baseline_scalers, X_sub)

DRP_TARGET = 'Dissolved Reactive Phosphorus'
drp_median = float(train_df[DRP_TARGET].median())
drp_q995 = float(train_df[DRP_TARGET].quantile(0.995))

drp_anchor = preds_anchor[DRP_TARGET]
drp_a50 = np.clip(DRP_ALPHAS[0] * drp_anchor + (1.0 - DRP_ALPHAS[0]) * drp_median, 0, drp_q995)
drp_a35 = np.clip(DRP_ALPHAS[1] * drp_anchor + (1.0 - DRP_ALPHAS[1]) * drp_median, 0, drp_q995)

submission_anchor = make_submission(
    preds_anchor['Total Alkalinity'],
    preds_anchor['Electrical Conductance'],
    drp_anchor,
)
submission_drp_a50 = make_submission(
    preds_anchor['Total Alkalinity'],
    preds_anchor['Electrical Conductance'],
    drp_a50,
)
submission_drp_a35 = make_submission(
    preds_anchor['Total Alkalinity'],
    preds_anchor['Electrical Conductance'],
    drp_a35,
)

delta_table = pd.DataFrame({
    'Variant': ['A_anchor', 'B_drp_shrink_a50', 'C_drp_shrink_a35'],
    'DRP_MAE_vs_anchor': [
        0.0,
        float(np.mean(np.abs(submission_drp_a50[DRP_TARGET] - submission_anchor[DRP_TARGET]))),
        float(np.mean(np.abs(submission_drp_a35[DRP_TARGET] - submission_anchor[DRP_TARGET]))),
    ],
    'DRP_Min': [
        float(submission_anchor[DRP_TARGET].min()),
        float(submission_drp_a50[DRP_TARGET].min()),
        float(submission_drp_a35[DRP_TARGET].min()),
    ],
    'DRP_Max': [
        float(submission_anchor[DRP_TARGET].max()),
        float(submission_drp_a50[DRP_TARGET].max()),
        float(submission_drp_a35[DRP_TARGET].max()),
    ],
})

submission_df = submission_drp_a50.copy()
delta_table


In [ ]:
os.makedirs('../data/submission', exist_ok=True)
stamp = datetime.now().strftime('%Y%m%d_%H%M')

path_a = f'../data/submission/submission_{stamp}_03c_A_anchor_rf_baseline.csv'
path_b = f'../data/submission/submission_{stamp}_03c_B_drp_shrink_a50.csv'
path_c = f'../data/submission/submission_{stamp}_03c_C_drp_shrink_a35.csv'

submission_anchor.to_csv(path_a, index=False)
submission_drp_a50.to_csv(path_b, index=False)
submission_drp_a35.to_csv(path_c, index=False)

print('Saved:')
print('A (anchor)      :', path_a)
print('B (DRP a=0.50)  :', path_b)
print('C (DRP a=0.35)  :', path_c)
